# 24 — PCA → WJ Simplex 512

Baseline for learned/nonlearned dimensionality reduction. It fits Incremental PCA on the corpus vectors, projects all vectors to 512 dimensions, shifts negatives to nonnegative values, L1-normalizes, then indexes with HNSW + WeightedJaccard.

This is an important sanity check: if PCA is close to the MLP, the neural part is less novel; if PCA is much worse, it supports the metric-aware learning story.


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup,
    eval_recall,
    l1_simplex,
    load_dataset,
    nmslib_neighbors,
    recall_at_k,
    rerank_raw_wj_numpy,
    save_result,
    shifted_l1_simplex,
)

# Edit here
dataset_name = "10k"       # "10k" or "full"
out_dim = 512
THREADS = 32
seed = 42
run_rerank = True
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]
np.random.seed(seed)

METHOD_NAME = "pca_wj_simplex_512"
NOTEBOOK_NAME = "24_pca_wj_simplex_512.ipynb"
OUT_PATH = "/tmp/results_sota_pca_wj_512.pkl"


In [ ]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
max_k = max(max(candidate_ks), 500)


In [ ]:
from sklearn.decomposition import IncrementalPCA

batch_size = 1024
fit_rows = min(len(corpus_qt), 50000)
print(f"Fitting IncrementalPCA on {fit_rows:,} corpus rows -> {out_dim} dims")
pca = IncrementalPCA(n_components=out_dim, batch_size=batch_size)
for start in range(0, fit_rows, batch_size):
    pca.partial_fit(corpus_qt[start:start + batch_size])

def transform_chunks(x):
    chunks = []
    for start in range(0, len(x), batch_size):
        chunks.append(pca.transform(x[start:start + batch_size]).astype(np.float32))
    return np.vstack(chunks)

z = transform_chunks(qt)
print(f"explained variance ratio sum={pca.explained_variance_ratio_.sum():.4f}")
embs = shifted_l1_simplex(z)


In [ ]:
corpus_embs = embs[:query_start]
query_embs = embs[query_start:]
print(f"embs={embs.shape} | simplex sums: {embs.sum(axis=1).min():.4f} .. {embs.sum(axis=1).max():.4f}")
print(f"corpus vector memory = {corpus_embs.nbytes / 1024**2:.1f} MB")

nbrs, ann_info = nmslib_neighbors(
    corpus_embs,
    query_embs,
    space="WeightedJaccard",
    k=max_k,
    threads=THREADS,
)
metrics = {
    **eval_recall(gt, nbrs, query_start, max_k),
    **ann_info,
    "dim": int(corpus_embs.shape[1]),
    "vec_mb": float(corpus_embs.nbytes / 1024**2),
}
print("\nNo rerank")
for k, v in metrics.items():
    if isinstance(k, int):
        print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={metrics['qps']:.1f}")
results = {METHOD_NAME: metrics}

if run_rerank:
    for ck in candidate_ks:
        print(f"\nRaw-WJ rerank from top-{ck}")
        cand, cand_info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_raw_wj_numpy(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time() - t0 + len(query_qt) / max(cand_info['qps'], 1e-9), 1e-9)
        rr_metrics = {
            **eval_recall(gt, rr, query_start, ck),
            "qps": qps_total,
            "qps_candidates": cand_info["qps"],
            "candidate_k": ck,
        }
        for k, v in rr_metrics.items():
            if isinstance(k, int):
                print(f"R@{k:<4} = {v:.4f}")
        print(f"QPS={qps_total:.1f}")
        results[f"{METHOD_NAME}_rerank_{ck}"] = rr_metrics

for key, value in results.items():
    save_result(OUT_PATH, dataset_name, key, value, meta={"notebook": NOTEBOOK_NAME})
cleanup()
